In [16]:
#Declare library
import os
import warnings

import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

warnings.filterwarnings("ignore")

In [17]:
core_cols = [
    'TransactionID', 'TransactionDT', 'TransactionAmt',
    'card1', 'card2', 'card3', 'card5',
    'addr1', 'addr2', 'dist1',
    # C columns — đếm số lần trùng giao dịch
    'C1', 'C2', 'C6', 'C11', 'C13', 'C14',
    # D columns — tối thiểu cần thiết
    'D1',   # bắt buộc để tạo UID
    'D10',  # time delta có correlation cao
    'D15',  # time delta có correlation cao
]

cat_cols = [
    'ProductCD', 'card4', 'card6',
    'P_emaildomain', 'R_emaildomain',
    'M4', 'M6',                          # giữ 2 M cols có signal cao nhất
    'id_30', 'id_31', 'DeviceType'        # device info có giá trị
]

v_no_nan = [1,3,4,6,8,11]
v_no_nan += [13,14,17,20,23,26,27,30]
v_no_nan += [36,37,40,41,44,47,48]
v_no_nan += [54,56,59,62,65,67,68,70]
v_no_nan += [76,78,80,82,86,88,89,91]
v_no_nan += [107,108,111,115,117,120,121,123]
v_no_nan += [124,127,129,130,136]
v_no_nan += [281,283,289,296,301,314]
v_no_nan += [294,284,285,286,291,297]
v_no_nan += [303,305,307,309,310,320]

v_cols = ['V' + str(x) for x in sorted(set(v_no_nan))]

transaction_cols_test = core_cols + cat_cols + v_cols

# Loại bỏ các cột không phải feature
exclude_cols = {
    'TransactionID', 'isFraud', 'uid', 'D1_norm',
    # Loại cột categorical gốc (đã được freq-encode)
    'ProductCD', 'card4', 'card6',
    'P_emaildomain', 'R_emaildomain',
    'M4', 'M6', 'DeviceType', 'id_30', 'id_31', 'DeviceInfo', 'id_33'
}


In [18]:
def _get_spark():
    spark = (
        SparkSession.builder
        .appName("FraudDetection_Predict")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    return spark

In [19]:
def _load_artifacts(model_path, artifacts_dir, spark):
    booster = xgb.Booster()
    booster.load_model(model_path)
    print(f"Model : {model_path}")

    uid_agg_path  = os.path.join(artifacts_dir, "uid_agg_sample.parquet")
    uid_agg_pd    = pd.read_parquet(uid_agg_path)
    uid_agg_spark = spark.createDataFrame(uid_agg_pd)
    print(f"uid_agg   : {uid_agg_path}  ({len(uid_agg_pd):,} UIDs)")

    freq_maps_path  = os.path.join(artifacts_dir, "freq_maps_sample.pkl")
    with open(freq_maps_path, "rb") as f:
        freq_maps_pd = pickle.load(f)

    freq_maps_spark = {}
    encode_cols     = []
    for col_name, freq_pd in freq_maps_pd.items():
        freq_maps_spark[col_name] = spark.createDataFrame(freq_pd)
        encode_cols.append(col_name)
    print(f"freq_maps : {freq_maps_path}  ({len(encode_cols)} cols)")

    return booster, uid_agg_spark, freq_maps_spark, encode_cols

In [20]:
def _build_features(df_spark, uid_agg_spark, freq_maps_spark, encode_cols):
    # UID
    df_spark = (
        df_spark
        .withColumn(
            "D1_norm",
            F.round(F.col("TransactionDT") / F.lit(86400) - F.col("D1"), scale=0),
        )
        .withColumn(
            "uid",
            F.concat_ws(
                "_",
                F.col("card1").cast("string"),
                F.col("addr1").cast("string"),
                F.col("D1_norm").cast("string"),
            ),
        )
    )

    # UID aggregated features
    df_spark = df_spark.join(uid_agg_spark, on="uid", how="left")
    df_spark = df_spark.fillna({"uid_amt_std": 0.0})

    # Frequency encoding
    for col_name in encode_cols:
        if col_name not in df_spark.columns:
            continue
        df_spark = df_spark.join(freq_maps_spark[col_name], on=col_name, how="left")
        df_spark = df_spark.fillna({f"{col_name}_freq": 0})

    # Extra features
    df_spark = (
        df_spark
        .withColumn("tx_hour",
            ((F.col("TransactionDT") / 3600) % 24).cast("int"))
        .withColumn("tx_day_of_week",
            ((F.col("TransactionDT") / 86400) % 7).cast("int"))
        .withColumn("log_TransactionAmt",
            F.log1p(F.col("TransactionAmt")))
        .withColumn("amt_to_uid_mean_ratio",
            F.when(
                F.col("uid_amt_mean").isNotNull() & (F.col("uid_amt_mean") > 0),
                F.col("TransactionAmt") / F.col("uid_amt_mean"),
            ).otherwise(F.lit(1.0)))
    )

    return df_spark


In [21]:
def predict_fraud(
    input_path,
    model_path     = "./best_xgb_model/best_booster_sample.ubj",
    artifacts_dir  = "./artifacts",
    threshold      = 0.5,
    output_path    = "./test_output.csv",
):
    """
    Dự đoán giao dịch gian lận — hoàn toàn độc lập với các cell khác.

    Parameters
    ----------
    input_path    : str   — CSV đầu vào (test_transaction.csv)
    model_path    : str   — best_booster.ubj
    artifacts_dir : str   — thư mục chứa uid_agg.parquet + freq_*.parquet
    threshold     : float — ngưỡng fraud (default 0.5)
    output_path   : str   — path lưu CSV kết quả (None = không lưu)

    Returns
    -------
    pandas.DataFrame: TransactionID | fraud_prob | is_fraud
    """
    print(f"{'='*55}")
    print(f"  FRAUD DETECTION PREDICTION")
    print(f"{'='*55}")

    spark = _get_spark()


    print("\nLoading artifacts ...")
    booster, uid_agg_spark, freq_maps_spark, encode_cols = _load_artifacts(
        model_path, artifacts_dir, spark
    )

    print("\nReading input ...")
    df_raw  = spark.read.csv(input_path, header=True, inferSchema=True)
    needed  = [c for c in transaction_cols_test if c in df_raw.columns]
    df_raw  = df_raw.select(needed)
    n_input = df_raw.count()
    print(f" Input     : {input_path}  ({n_input:,} rows, {len(needed)} cols)")

    print("\nBuilding features ...")
    df_feat = _build_features(df_raw, uid_agg_spark, freq_maps_spark, encode_cols)

    feature_cols = [
        field.name for field in df_feat.schema.fields
        if field.name not in exclude_cols
        and isinstance(field.dataType, NumericType)
    ]
    print(f" Features  : {len(feature_cols)} columns")

    print("\nConverting to pandas ...")
    df_pd = df_feat.select(["TransactionID"] + feature_cols).toPandas()

    print("Predicting ...")
    X          = df_pd[feature_cols].values.astype(np.float32)
    dmatrix    = xgb.DMatrix(X, feature_names=feature_cols)
    fraud_prob = booster.predict(dmatrix)
    is_fraud   = (fraud_prob >= threshold).astype(int)

    result_df = pd.DataFrame({
        "TransactionID": df_pd["TransactionID"],
        "fraud_prob"   : np.round(fraud_prob, 4),
        "is_fraud"     : is_fraud,
    })

    n_total   = len(result_df)
    n_fraud   = int(is_fraud.sum())
    high_risk = int((fraud_prob >= 0.8).sum())

    print(f"\n{'='*55}")
    print(f"  KẾT QUẢ DỰ ĐOÁN  (threshold = {threshold})")
    print(f"{'='*55}")
    print(f"  Total transactions : {n_total:,}")
    print(f"  Fraud detected     : {n_fraud:,}  ({n_fraud/n_total*100:.2f}%)")
    print(f"  High risk (≥0.8)   : {high_risk:,}")
    print(f"  Avg fraud prob     : {fraud_prob.mean():.4f}")
    print(f"{'='*55}")

    if output_path:
        result_df.to_csv(output_path, index=False)
        print(f"  Saved → {output_path}")

    return result_df

In [22]:
result = predict_fraud(
    input_path    = "./test_transaction.csv",
    model_path    = "./best_xgb_model/best_booster_sample.ubj",
    artifacts_dir = "./artifacts",
    threshold     = 0.5,
    output_path   = "./test_output.csv",
)

result.sort_values("fraud_prob", ascending=False).head(20)

  FRAUD DETECTION PREDICTION

Loading artifacts ...
Model : ./best_xgb_model/best_booster_sample.ubj
uid_agg   : ./artifacts/uid_agg_sample.parquet  (7,107 UIDs)
freq_maps : ./artifacts/freq_maps_sample.pkl  (10 cols)

Reading input ...


 Input     : ./test_transaction.csv  (506,691 rows, 94 cols)

Building features ...
 Features  : 104 columns

Converting to pandas ...


Predicting ...

  KẾT QUẢ DỰ ĐOÁN  (threshold = 0.5)
  Total transactions : 506,691
  Fraud detected     : 190,186  (37.53%)
  High risk (≥0.8)   : 0
  Avg fraud prob     : 0.4896
  Saved → ./test_output.csv


,TransactionID,fraud_prob,is_fraud
81646,3745195,0.6118,1
275846,3939395,0.6118,1
445973,4109522,0.6118,1
445975,4109524,0.6118,1
275835,3939384,0.6118,1
201552,3865101,0.6118,1
445977,4109526,0.6118,1
275839,3939388,0.6118,1
66983,3730532,0.6118,1
275844,3939393,0.6118,1
